# 06 · 학습된 모델 시험하기
교통량·시드·제어기를 고른 뒤 **새 실험 → 5초 진행** 또는 **끝까지 실행**을 누르세요. SUMO의 실제 차량 이동을 표시합니다. GUI 화면 없이 Colab에서 사용할 수 있습니다. 새 실험은 보고서의 최종 평가와 별도로 저장합니다.

In [ ]:
from pathlib import Path
import os, sys, subprocess
candidates = [Path('/content/T_Junction_RL'), Path.cwd(), Path.cwd().parent,
              Path('/content/drive/MyDrive/T_Junction_RL')]
PROJECT_ROOT = next((p for p in candidates if (p/'REVISION_V2').exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('먼저 00_colab_setup 노트북에서 ZIP 업로드 및 설치를 실행하세요.')
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
print('프로젝트:', PROJECT_ROOT)

In [ ]:
from IPython.display import display,clear_output,Video,Image
from io import BytesIO
import ipywidgets as widgets
import matplotlib.pyplot as plt
from datetime import datetime
from src.playground import Playground
if Path('/content').exists():
    from google.colab import output
    output.enable_custom_widget_manager()
model=widgets.Text(value=str(globals().get('TRAIN_DIR',Path('outputs/revision/seed42'))/'checkpoints/best_model.pt'),description='모델:',layout=widgets.Layout(width='95%'))
from src.utils import load_yaml
scenario=widgets.Dropdown(options=list(load_yaml('scenarios.yaml')['scenarios']),value='high',description='교통량:')
seed=widgets.IntText(value=3050,description='Seed:')
horizon=widgets.IntSlider(value=900,min=60,max=3600,step=60,description='시간(초):')
controller=widgets.Dropdown(options=['dqn','fixed_time','rule_based'],description='제어기:')
start=widgets.Button(description='새 실험',button_style='primary')
step=widgets.Button(description='5초 진행')
finish=widgets.Button(description='끝까지 실행')
video=widgets.Button(description='영상 생성')
canvas=widgets.Output()
session=None

def show():
    with canvas:
        clear_output(wait=True)
        figure=session.figure()
        buffer=BytesIO();figure.savefig(buffer,format='png',dpi=120)
        display(Image(data=buffer.getvalue()));plt.close('all')
        if session.last_prediction: print(session.last_prediction)
        if session.done: print('완료:',session.info['metrics'])

def begin(_):
    global session
    try:
        if session: session.close()
        path=Path('outputs/playground')/datetime.now().strftime('%Y%m%d_%H%M%S_%f')
        session=Playground(model.value,scenario.value,seed.value,horizon.value,controller.value,path)
        show()
    except Exception as e:
        with canvas: print(type(e).__name__,str(e))

def advance(_):
    if session:
        session.advance(5);show()

def complete(_):
    if session:
        finish.disabled=True
        try:
            while not session.done:
                session.advance(50);show()
        finally: finish.disabled=False

def encode(_):
    if session:
        with canvas:
            try: display(Video(str(session.video()),embed=True))
            except Exception as e: print(str(e))

start.on_click(begin);step.on_click(advance);finish.on_click(complete);video.on_click(encode)
display(model,widgets.HBox([scenario,controller]),widgets.HBox([seed,horizon]),widgets.HBox([start,step,finish,video]),canvas)